In [ ]:
from google.colab import drive
import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

raw_10_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_raw.parquet"
    )
)

raw_10_df["Datetime"] = pd.to_datetime(
    raw_10_df["Datetime"]
)

raw_10_df["StationId"] = (
    raw_10_df["StationId"]
    .astype(str)
)

raw_10_df = raw_10_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("=" * 60)
print("10-STATION DATASET RESTORED")
print("=" * 60)

print("Records:", len(raw_10_df))
print("Stations:", raw_10_df["StationId"].nunique())

print(
    "Start:",
    raw_10_df["Datetime"].min()
)

print(
    "End:",
    raw_10_df["Datetime"].max()
)

In [ ]:
RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

BASE_COLUMNS = [
    "StationId",
    "StationName",
    "Datetime"
] + RAW_FEATURES

TARGET = "AQI_t_plus_1"

print("Raw predictive measurements:")
print(RAW_FEATURES)

In [ ]:
DIWALI_DATES = pd.to_datetime([
    "2015-11-11",
    "2016-10-30",
    "2017-10-19",
    "2018-11-07",
    "2019-10-27",
    "2020-11-14"
])

def is_diwali_window(
    timestamp,
    days_before=2,
    days_after=2
):
    timestamp = pd.Timestamp(timestamp)

    return int(
        any(
            (
                timestamp >= date - pd.Timedelta(
                    days=days_before
                )
            )
            and
            (
                timestamp <= date + pd.Timedelta(
                    days=days_after
                )
            )
            for date in DIWALI_DATES
        )
    )

print("Calendar event logic ready.")

In [ ]:
def build_canonical_features(
    history_df
):
    """
    Build model input features using only information
    available at each timestamp.

    The same function is used for:
      1. historical model-training data
      2. historical live-style inference
      3. future CPCB/OGD inference
    """

    df = history_df[
        BASE_COLUMNS
    ].copy()

    df["Datetime"] = pd.to_datetime(
        df["Datetime"]
    )

    df["StationId"] = (
        df["StationId"]
        .astype(str)
    )

    df = df.sort_values(
        ["StationId", "Datetime"]
    ).reset_index(drop=True)

    output_parts = []

    # ----------------------------------------------
    # Process each station independently
    # ----------------------------------------------

    for station_id, station_group in df.groupby(
        "StationId",
        sort=False
    ):

        station = (
            station_group
            .sort_values("Datetime")
            .copy()
        )

        # ------------------------------------------
        # Basic time features
        # ------------------------------------------

        station["Hour"] = (
            station["Datetime"].dt.hour
        )

        station["Day"] = (
            station["Datetime"].dt.day
        )

        station["Month"] = (
            station["Datetime"].dt.month
        )

        station["Year"] = (
            station["Datetime"].dt.year
        )

        station["DayOfWeek"] = (
            station["Datetime"].dt.dayofweek
        )

        station["IsWeekend"] = (
            station["DayOfWeek"] >= 5
        ).astype("int8")

        station["IsStubbleSeason"] = (
            station["Month"].isin([10, 11])
        ).astype("int8")

        station["IsDiwaliWindow"] = (
            station["Datetime"]
            .apply(is_diwali_window)
            .astype("int8")
        )

        # ------------------------------------------
        # Exact timestamp-based lag features
        # ------------------------------------------

        lag_source = station[
            ["Datetime"] + RAW_FEATURES
        ].copy()

        for lag in [1, 2, 12]:

            shifted = lag_source.copy()

            shifted["Datetime"] = (
                shifted["Datetime"]
                + pd.Timedelta(
                    hours=lag
                )
            )

            rename_map = {
                column: f"{column}_lag_{lag}h"
                for column in RAW_FEATURES
            }

            shifted = shifted.rename(
                columns=rename_map
            )

            station = station.merge(
                shifted,
                on="Datetime",
                how="left",
                validate="one_to_one"
            )

        # ------------------------------------------
        # Rolling statistics
        # ------------------------------------------

        rolling_source = (
            station[
                [
                    "Datetime",
                    "AQI",
                    "PM2.5",
                    "PM10",
                    "NO2"
                ]
            ]
            .set_index("Datetime")
            .sort_index()
        )

        for column in [
            "AQI",
            "PM2.5",
            "PM10",
            "NO2"
        ]:

            station[
                f"{column}_rolling_mean_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

            station[
                f"{column}_rolling_mean_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

        # ------------------------------------------
        # Exact t+1 target
        # ------------------------------------------

        target_source = station[
            [
                "Datetime",
                "AQI"
            ]
        ].copy()

        target_source["TargetTime"] = (
            target_source["Datetime"]
            - pd.Timedelta(
                hours=1
            )
        )

        target_source = target_source.rename(
            columns={
                "AQI": TARGET,
                "TargetTime": "Datetime"
            }
        )

        station = station.merge(
            target_source[
                [
                    "Datetime",
                    TARGET
                ]
            ],
            on="Datetime",
            how="left",
            validate="one_to_one"
        )

        output_parts.append(
            station
        )

    result = pd.concat(
        output_parts,
        ignore_index=True
    )

    return result.sort_values(
        [
            "StationId",
            "Datetime"
        ]
    ).reset_index(drop=True)

In [ ]:
def build_canonical_features(
    history_df
):
    """
    Canonical t -> t+1 feature builder.

    The same function is intended for:
      1. historical model training
      2. historical live-style testing
      3. future CPCB/OGD inference

    All predictors for timestamp t use information
    available at or before t.
    """

    df = history_df[
        BASE_COLUMNS
    ].copy()

    df["Datetime"] = pd.to_datetime(
        df["Datetime"]
    )

    df["StationId"] = (
        df["StationId"]
        .astype(str)
    )

    df = (
        df
        .sort_values(
            ["StationId", "Datetime"]
        )
        .drop_duplicates(
            ["StationId", "Datetime"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    output_parts = []

    for station_id, station_group in df.groupby(
        "StationId",
        sort=False
    ):

        station = (
            station_group
            .sort_values("Datetime")
            .copy()
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # Calendar features
        # --------------------------------------------------

        station["Hour"] = (
            station["Datetime"].dt.hour
        )

        station["Day"] = (
            station["Datetime"].dt.day
        )

        station["Month"] = (
            station["Datetime"].dt.month
        )

        station["Year"] = (
            station["Datetime"].dt.year
        )

        station["DayOfWeek"] = (
            station["Datetime"].dt.dayofweek
        )

        station["IsWeekend"] = (
            station["DayOfWeek"] >= 5
        ).astype("int8")

        station["IsStubbleSeason"] = (
            station["Month"].isin([10, 11])
        ).astype("int8")

        station["IsDiwaliWindow"] = (
            station["Datetime"]
            .apply(is_diwali_window)
            .astype("int8")
        )

        # --------------------------------------------------
        # Exact timestamp lags
        # --------------------------------------------------

        for lag in [1, 2, 12]:

            lag_source = station[
                ["Datetime"] + RAW_FEATURES
            ].copy()

            lag_source["Datetime"] = (
                lag_source["Datetime"]
                + pd.Timedelta(hours=lag)
            )

            lag_source = lag_source.rename(
                columns={
                    column: f"{column}_lag_{lag}h"
                    for column in RAW_FEATURES
                }
            )

            station = station.merge(
                lag_source[
                    ["Datetime"]
                    + [
                        f"{column}_lag_{lag}h"
                        for column in RAW_FEATURES
                    ]
                ],
                on="Datetime",
                how="left",
                validate="one_to_one"
            )

        # --------------------------------------------------
        # Rolling features
        # --------------------------------------------------

        rolling_source = (
            station[
                [
                    "Datetime",
                    "AQI",
                    "PM2.5",
                    "PM10",
                    "NO2"
                ]
            ]
            .drop_duplicates("Datetime")
            .set_index("Datetime")
            .sort_index()
        )

        for column in [
            "AQI",
            "PM2.5",
            "PM10",
            "NO2"
        ]:

            station[
                f"{column}_rolling_mean_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

            station[
                f"{column}_rolling_mean_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

        # --------------------------------------------------
        # Exact t+1 target
        # --------------------------------------------------

        # Original AQI at t+1 becomes the target
        # attached to timestamp t.
        future_aqi = station[
            ["Datetime", "AQI"]
        ].copy()

        future_aqi["Datetime"] = (
            future_aqi["Datetime"]
            - pd.Timedelta(hours=1)
        )

        future_aqi = future_aqi.rename(
            columns={
                "AQI": TARGET
            }
        )

        future_aqi = future_aqi[
            ["Datetime", TARGET]
        ]

        station = station.merge(
            future_aqi,
            on="Datetime",
            how="left",
            validate="one_to_one"
        )

        output_parts.append(
            station
        )

    result = pd.concat(
        output_parts,
        ignore_index=True
    )

    # Final safety checks
    if result.columns.duplicated().any():

        duplicated_columns = (
            result.columns[
                result.columns.duplicated()
            ]
            .tolist()
        )

        raise ValueError(
            "Duplicate columns remain: "
            f"{duplicated_columns}"
        )

    return (
        result
        .sort_values(
            ["StationId", "Datetime"]
        )
        .reset_index(drop=True)
    )


print("Canonical feature builder updated successfully.")

In [ ]:
canonical_df = build_canonical_features(
    raw_10_df
)

print("=" * 60)
print("CANONICAL FEATURE DATASET")
print("=" * 60)

print(
    "Rows:",
    len(canonical_df)
)

print(
    "Stations:",
    canonical_df["StationId"].nunique()
)

print(
    "Start:",
    canonical_df["Datetime"].min()
)

print(
    "End:",
    canonical_df["Datetime"].max()
)

print(
    "Target column:",
    TARGET
)

print(
    "Target missing:",
    canonical_df[TARGET].isna().sum()
)

print(
    "Duplicate Station-Time:",
    canonical_df.duplicated(
        ["StationId", "Datetime"]
    ).sum()
)

print(
    "Duplicate columns:",
    canonical_df.columns.duplicated().sum()
)

In [ ]:
# ============================================================
# STEP 12.0.6 — FINAL MODEL FEATURE CONTRACT
# ============================================================

RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

# Current-time features
current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

# Lag features
lag_features = []

for lag in [1, 2, 12]:
    for column in RAW_FEATURES:
        lag_features.append(
            f"{column}_lag_{lag}h"
        )

# Rolling features
rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

# Calendar features
calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

# Event features
event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

# Spatial feature
categorical_features = [
    "StationId"
]

# Complete model feature contract
numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

model_features = (
    numeric_features
    + categorical_features
)

# ------------------------------------------------------------
# Verify against canonical dataset
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in model_features
    if feature not in canonical_df.columns
]

print("=" * 60)
print("10-STATION MODEL FEATURE CONTRACT")
print("=" * 60)

print("Current features :", len(current_features))
print("Lag features     :", len(lag_features))
print("Rolling features :", len(rolling_features))
print("Calendar features:", len(calendar_features))
print("Event features   :", len(event_features))
print("Station features :", len(categorical_features))
print("Total features   :", len(model_features))

print("\nMissing features:")
print(missing_features)

print(
    "\nSeason included:",
    "Season" in model_features
)

# Hard validation
assert len(missing_features) == 0, (
    f"Missing required features: {missing_features}"
)

assert "Season" not in model_features, (
    "Season must not be part of the Random Forest feature contract."
)

print("\nFEATURE CONTRACT VERIFIED")

In [ ]:
# ============================================================
# STEP 12.0.7 — DATASET INTEGRITY CHECK
# ============================================================

print("=" * 60)
print("CANONICAL DATASET INTEGRITY")
print("=" * 60)

print(
    "Rows:",
    len(canonical_df)
)

print(
    "Stations:",
    canonical_df["StationId"].nunique()
)

print(
    "Missing target:",
    canonical_df[TARGET].isna().sum()
)

print(
    "Negative target:",
    (
        canonical_df[TARGET]
        .dropna()
        .lt(0)
        .sum()
    )
)

print(
    "Duplicate Station-Time:",
    canonical_df.duplicated(
        ["StationId", "Datetime"]
    ).sum()
)

print(
    "Duplicate columns:",
    canonical_df.columns.duplicated().sum()
)

print(
    "Datetime dtype:",
    canonical_df["Datetime"].dtype
)

In [ ]:
# ============================================================
# STEP 12.0.8 — FINAL SUPERVISED DATASET
# ============================================================

model_df = (
    canonical_df[
        canonical_df[TARGET].notna()
    ]
    .copy()
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL 10-STATION SUPERVISED DATASET")
print("=" * 60)

print("Rows:", len(model_df))
print(
    "Stations:",
    model_df["StationId"].nunique()
)

print(
    "Target missing:",
    model_df[TARGET].isna().sum()
)

print(
    "Start:",
    model_df["Datetime"].min()
)

print(
    "End:",
    model_df["Datetime"].max()
)

In [ ]:
# ============================================================
# STEP 12.0.9 — CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

import numpy as np
import pandas as pd

# Ensure datetime is correct
model_df["Datetime"] = pd.to_datetime(
    model_df["Datetime"]
)

# Get unique timestamps
unique_times = np.sort(
    model_df["Datetime"].unique()
)

# 80% historical training / 20% future testing
split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

train_10 = (
    model_df[
        model_df["Datetime"] <= train_end_time
    ]
    .copy()
    .reset_index(drop=True)
)

test_10 = (
    model_df[
        model_df["Datetime"] > train_end_time
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 60)
print("10-STATION CHRONOLOGICAL SPLIT")
print("=" * 60)

print("TRAIN")
print(
    "Start:",
    train_10["Datetime"].min()
)

print(
    "End:",
    train_10["Datetime"].max()
)

print(
    "Rows:",
    len(train_10)
)

print("\nTEST")
print(
    "Start:",
    test_10["Datetime"].min()
)

print(
    "End:",
    test_10["Datetime"].max()
)

print(
    "Rows:",
    len(test_10)
)

print(
    "\nTemporal separation:",
    train_10["Datetime"].max()
    < test_10["Datetime"].min()
)

print(
    "\nTrain stations:",
    train_10["StationId"].nunique()
)

print(
    "Test stations:",
    test_10["StationId"].nunique()
)

In [ ]:
# ============================================================
# STEP 12.0.10 — STATION COVERAGE CHECK
# ============================================================

train_stations = set(
    train_10["StationId"].astype(str).unique()
)

test_stations = set(
    test_10["StationId"].astype(str).unique()
)

print("=" * 60)
print("STATION COVERAGE")
print("=" * 60)

print(
    "Training stations:",
    len(train_stations)
)

print(
    "Test stations:",
    len(test_stations)
)

print(
    "Stations missing from test:",
    sorted(
        train_stations - test_stations
    )
)

print(
    "Stations missing from train:",
    sorted(
        test_stations - train_stations
    )
)

assert len(train_stations) == 10, (
    "Training set does not contain all 10 stations."
)

assert len(test_stations) == 10, (
    "Test set does not contain all 10 stations."
)

print("\nAll 10 stations present in both periods.")

In [ ]:
# ============================================================
# STEP 12.0.11 — RANDOM FOREST PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

# These variables were defined in Step 12.0.6.
# Re-validate them here before model creation.
assert len(model_features) > 0
assert len(numeric_features) > 0
assert categorical_features == ["StationId"]

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor_10 = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

rf_model_10 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_10
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                max_depth=18,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline ready.")

In [ ]:
# ============================================================
# STEP 12.0.12 — TRAIN 10-STATION RANDOM FOREST
# ============================================================

X_train_10 = train_10[
    model_features
].copy()

y_train_10 = train_10[
    TARGET
].copy()

X_test_10 = test_10[
    model_features
].copy()

y_test_10 = test_10[
    TARGET
].copy()

print(
    "Training matrix:",
    X_train_10.shape
)

print(
    "Test matrix:",
    X_test_10.shape
)

rf_model_10.fit(
    X_train_10,
    y_train_10
)

print("\n10-station Random Forest training completed.")

In [ ]:
# ============================================================
# STEP 12.0.13 — RANDOM FOREST EVALUATION
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

rf_pred_10 = rf_model_10.predict(
    X_test_10
)

rf_mae_10 = mean_absolute_error(
    y_test_10,
    rf_pred_10
)

rf_rmse_10 = np.sqrt(
    mean_squared_error(
        y_test_10,
        rf_pred_10
    )
)

rf_r2_10 = r2_score(
    y_test_10,
    rf_pred_10
)

print("=" * 60)
print("10-STATION RANDOM FOREST RESULTS")
print("=" * 60)

print(
    "MAE :",
    round(rf_mae_10, 6)
)

print(
    "RMSE:",
    round(rf_rmse_10, 6)
)

print(
    "R²  :",
    round(rf_r2_10, 6)
)

In [ ]:
# ============================================================
# STEP 12.0.14 — FIXED PERSISTENCE BASELINE
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Persistence prediction:
# AQI(t+1) prediction = AQI(t)

baseline_df = test_10[
    [
        "StationId",
        "Datetime",
        "AQI",
        TARGET
    ]
].copy()

# Keep only rows where BOTH current AQI and
# actual next-hour AQI are available.
baseline_df = baseline_df.dropna(
    subset=[
        "AQI",
        TARGET
    ]
).reset_index(drop=True)

baseline_pred_10 = (
    baseline_df["AQI"]
    .astype(float)
)

baseline_actual_10 = (
    baseline_df[TARGET]
    .astype(float)
)

baseline_mae_10 = mean_absolute_error(
    baseline_actual_10,
    baseline_pred_10
)

baseline_rmse_10 = np.sqrt(
    mean_squared_error(
        baseline_actual_10,
        baseline_pred_10
    )
)

baseline_r2_10 = r2_score(
    baseline_actual_10,
    baseline_pred_10
)

print("=" * 60)
print("10-STATION PERSISTENCE BASELINE")
print("=" * 60)

print(
    "Usable baseline records:",
    len(baseline_df)
)

print(
    "MAE :",
    round(baseline_mae_10, 6)
)

print(
    "RMSE:",
    round(baseline_rmse_10, 6)
)

print(
    "R²  :",
    round(baseline_r2_10, 6)
)

print(
    "\nExcluded rows with missing current AQI:",
    len(test_10) - len(baseline_df)
)

In [ ]:
# ============================================================
# STEP 12.0.15 — CORRECTED BASELINE IMPROVEMENT
# ============================================================

mae_improvement_10 = (
    (
        baseline_mae_10
        - rf_mae_10
    )
    / baseline_mae_10
) * 100

rmse_improvement_10 = (
    (
        baseline_rmse_10
        - rf_rmse_10
    )
    / baseline_rmse_10
) * 100

print(
    "Random Forest MAE improvement vs persistence:",
    round(
        mae_improvement_10,
        4
    ),
    "%"
)

print(
    "Random Forest RMSE improvement vs persistence:",
    round(
        rmse_improvement_10,
        4
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 12.0.15 — BASELINE IMPROVEMENT
# ============================================================

mae_improvement_10 = (
    (
        baseline_mae_10
        - rf_mae_10
    )
    / baseline_mae_10
) * 100

rmse_improvement_10 = (
    (
        baseline_rmse_10
        - rf_rmse_10
    )
    / baseline_rmse_10
) * 100

print(
    "MAE improvement vs persistence:",
    round(
        mae_improvement_10,
        4
    ),
    "%"
)

print(
    "RMSE improvement vs persistence:",
    round(
        rmse_improvement_10,
        4
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 12.0.16 — CANONICAL LIVE-STYLE INFERENCE
# ============================================================

def build_live_input_10(
    raw_history,
    station_id,
    prediction_time
):
    prediction_time = pd.Timestamp(
        prediction_time
    )

    station_history = (
        raw_history[
            (
                raw_history["StationId"]
                .astype(str)
                == str(station_id)
            )
            &
            (
                raw_history["Datetime"]
                <= prediction_time
            )
        ]
        .copy()
        .sort_values("Datetime")
    )

    if len(station_history) == 0:
        raise ValueError(
            "No historical data available "
            "for this station."
        )

    canonical_live = (
        build_canonical_features(
            station_history
        )
    )

    current_row = canonical_live[
        canonical_live["Datetime"]
        == prediction_time
    ].copy()

    if len(current_row) != 1:
        raise ValueError(
            "Current timestamp is not available "
            "exactly once."
        )

    current_row = current_row[
        model_features
    ].copy()

    return current_row

In [ ]:
# ============================================================
# STEP 12.0.17 — SINGLE CASE PARITY TEST
# ============================================================

parity_station = str(
    test_10["StationId"].iloc[0]
)

parity_time = pd.Timestamp(
    test_10[
        test_10["StationId"].astype(str)
        == parity_station
    ]["Datetime"].iloc[0]
)

training_row = (
    model_df[
        (
            model_df["StationId"].astype(str)
            == parity_station
        )
        &
        (
            model_df["Datetime"]
            == parity_time
        )
    ][
        model_features
    ]
    .copy()
)

live_row = build_live_input_10(
    raw_10_df,
    parity_station,
    parity_time
)

print(
    "Station:",
    parity_station
)

print(
    "Prediction time:",
    parity_time
)

print(
    "Training row shape:",
    training_row.shape
)

print(
    "Live row shape:",
    live_row.shape
)

In [ ]:
# ============================================================
# STEP 12.0.18 — FEATURE PARITY
# ============================================================

parity_rows = []

for feature in numeric_features:

    training_value = training_row.iloc[0][
        feature
    ]

    live_value = live_row.iloc[0][
        feature
    ]

    if (
        pd.isna(training_value)
        and pd.isna(live_value)
    ):
        difference = 0.0

    elif (
        pd.isna(training_value)
        or pd.isna(live_value)
    ):
        difference = np.inf

    else:
        difference = abs(
            float(training_value)
            - float(live_value)
        )

    parity_rows.append(
        {
            "Feature": feature,
            "TrainingValue": training_value,
            "LiveValue": live_value,
            "AbsoluteDifference": difference
        }
    )

single_parity = pd.DataFrame(
    parity_rows
)

finite_parity = single_parity[
    np.isfinite(
        single_parity[
            "AbsoluteDifference"
        ]
    )
]

print(
    "Numeric features checked:",
    len(single_parity)
)

print(
    "Exact matches:",
    (
        finite_parity[
            "AbsoluteDifference"
        ] <= 1e-9
    ).sum()
)

print(
    "Mismatches:",
    (
        finite_parity[
            "AbsoluteDifference"
        ] > 1e-9
    ).sum()
)

print(
    "Maximum difference:",
    finite_parity[
        "AbsoluteDifference"
    ].max()
)

display(
    single_parity[
        single_parity[
            "AbsoluteDifference"
        ] > 1e-9
    ]
)

In [ ]:
# ============================================================
# STEP 12.0.19 — PREDICTION PARITY
# ============================================================

training_prediction = float(
    rf_model_10.predict(
        training_row
    )[0]
)

live_prediction = float(
    rf_model_10.predict(
        live_row
    )[0]
)

prediction_difference = abs(
    training_prediction
    - live_prediction
)

print(
    "Training-style prediction:",
    round(
        training_prediction,
        10
    )
)

print(
    "Live-style prediction:",
    round(
        live_prediction,
        10
    )
)

print(
    "Difference:",
    round(
        prediction_difference,
        12
    )
)

In [ ]:
# ============================================================
# STEP 12.0.20 — MULTI-CASE PARITY AUDIT
# ============================================================

rng = np.random.default_rng(
    42
)

eligible_cases = (
    test_10[
        [
            "StationId",
            "Datetime"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

sample_count = min(
    100,
    len(eligible_cases)
)

sample_indices = rng.choice(
    len(eligible_cases),
    size=sample_count,
    replace=False
)

parity_results = []

for index in sample_indices:

    station_id = str(
        eligible_cases.iloc[index][
            "StationId"
        ]
    )

    prediction_time = pd.Timestamp(
        eligible_cases.iloc[index][
            "Datetime"
        ]
    )

    training_row = (
        model_df[
            (
                model_df["StationId"].astype(str)
                == station_id
            )
            &
            (
                model_df["Datetime"]
                == prediction_time
            )
        ][
            model_features
        ]
        .copy()
    )

    if len(training_row) != 1:
        continue

    try:
        live_row = build_live_input_10(
            raw_10_df,
            station_id,
            prediction_time
        )

        training_pred = float(
            rf_model_10.predict(
                training_row
            )[0]
        )

        live_pred = float(
            rf_model_10.predict(
                live_row
            )[0]
        )

        parity_results.append(
            {
                "StationId": station_id,
                "Datetime": prediction_time,
                "TrainingPrediction": training_pred,
                "LiveStylePrediction": live_pred,
                "PredictionDifference": abs(
                    training_pred
                    - live_pred
                )
            }
        )

    except Exception as e:
        parity_results.append(
            {
                "StationId": station_id,
                "Datetime": prediction_time,
                "TrainingPrediction": np.nan,
                "LiveStylePrediction": np.nan,
                "PredictionDifference": np.nan
            }
        )

parity_results = pd.DataFrame(
    parity_results
)

valid_parity = parity_results[
    parity_results[
        "PredictionDifference"
    ].notna()
]

print(
    "Cases tested:",
    len(parity_results)
)

print(
    "Valid cases:",
    len(valid_parity)
)

print(
    "Maximum prediction difference:",
    valid_parity[
        "PredictionDifference"
    ].max()
)

print(
    "Mean prediction difference:",
    valid_parity[
        "PredictionDifference"
    ].mean()
)

In [ ]:
# ============================================================
# STEP 12.0.21 — LIVE-STYLE HISTORICAL EVALUATION
# ============================================================

live_eval_rows = []

for index in sample_indices:

    station_id = str(
        eligible_cases.iloc[index][
            "StationId"
        ]
    )

    prediction_time = pd.Timestamp(
        eligible_cases.iloc[index][
            "Datetime"
        ]
    )

    actual_row = test_10[
        (
            test_10["StationId"].astype(str)
            == station_id
        )
        &
        (
            test_10["Datetime"]
            == prediction_time
        )
    ]

    if len(actual_row) != 1:
        continue

    try:

        live_row = build_live_input_10(
            raw_10_df,
            station_id,
            prediction_time
        )

        prediction = float(
            rf_model_10.predict(
                live_row
            )[0]
        )

        actual = float(
            actual_row[
                TARGET
            ].iloc[0]
        )

        live_eval_rows.append(
            {
                "StationId": station_id,
                "Datetime": prediction_time,
                "Predicted_AQI_t_plus_1": prediction,
                "Actual_AQI_t_plus_1": actual,
                "AbsoluteError": abs(
                    actual - prediction
                )
            }
        )

    except Exception:
        continue

live_eval = pd.DataFrame(
    live_eval_rows
)

live_style_mae = mean_absolute_error(
    live_eval[
        "Actual_AQI_t_plus_1"
    ],
    live_eval[
        "Predicted_AQI_t_plus_1"
    ]
)

live_style_rmse = np.sqrt(
    mean_squared_error(
        live_eval[
            "Actual_AQI_t_plus_1"
        ],
        live_eval[
            "Predicted_AQI_t_plus_1"
        ]
    )
)

live_style_r2 = r2_score(
    live_eval[
        "Actual_AQI_t_plus_1"
    ],
    live_eval[
        "Predicted_AQI_t_plus_1"
    ]
)

print("=" * 60)
print("LIVE-STYLE HISTORICAL EVALUATION")
print("=" * 60)

print(
    "Cases:",
    len(live_eval)
)

print(
    "MAE:",
    round(
        live_style_mae,
        6
    )
)

print(
    "RMSE:",
    round(
        live_style_rmse,
        6
    )
)

print(
    "R²:",
    round(
        live_style_r2,
        6
    )
)

In [ ]:
# ============================================================
# STEP 12.0.22 — SAVE FINAL 10-STATION MODEL ARTIFACTS
# ============================================================

canonical_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_10_station_feature_engineered.parquet"
)

model_df.to_parquet(
    canonical_dataset_path,
    index=False
)

rf_model_10_path = os.path.join(
    MODEL_PATH,
    "final_10_station_random_forest.joblib"
)

joblib.dump(
    rf_model_10,
    rf_model_10_path
)

print(
    "Canonical dataset saved:",
    canonical_dataset_path
)

print(
    "Final 10-station RF saved:",
    rf_model_10_path
)

In [ ]:
# ============================================================
# STEP 12.0.23 — SAVE AUDIT REPORTS
# ============================================================

single_parity.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase12_0_single_case_feature_parity.csv"
    ),
    index=False
)

parity_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase12_0_prediction_parity.csv"
    ),
    index=False
)

live_eval.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase12_0_live_style_evaluation.csv"
    ),
    index=False
)

print("Phase 12.0 audit reports saved.")

In [ ]:
# ============================================================
# STEP 12.0.24 — PHASE 12.0 FINAL SUMMARY
# ============================================================

parity_max_difference = (
    valid_parity[
        "PredictionDifference"
    ].max()
)

parity_passed = (
    len(valid_parity) > 0
    and parity_max_difference <= 1e-8
)

print("=" * 70)
print("PHASE 12.0 — 10-STATION PARITY AUDIT")
print("=" * 70)

print(
    "Stations:",
    model_df["StationId"].nunique()
)

print(
    "Model rows:",
    len(model_df)
)

print("\n10-STATION RANDOM FOREST")
print(
    "Offline MAE:",
    round(
        rf_mae_10,
        6
    )
)

print(
    "Offline RMSE:",
    round(
        rf_rmse_10,
        6
    )
)

print(
    "Offline R²:",
    round(
        rf_r2_10,
        6
    )
)

print("\nLIVE-STYLE HISTORICAL TEST")
print(
    "MAE:",
    round(
        live_style_mae,
        6
    )
)

print(
    "RMSE:",
    round(
        live_style_rmse,
        6
    )
)

print(
    "R²:",
    round(
        live_style_r2,
        6
    )
)

print("\nPREDICTION PARITY")
print(
    "Cases:",
    len(valid_parity)
)

print(
    "Maximum difference:",
    round(
        parity_max_difference,
        12
    )
)

print(
    "Parity passed:",
    parity_passed
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 12.0.25 — SAVE STATUS
# ============================================================

phase12_status = {
    "completed_phase": (
        "Phase 12.0 - 10-Station Training/Inference Parity Audit"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
        if parity_passed
        else
        "Phase 12.0 - Fix Parity Before Live Integration"
    ),

    "station_count": int(
        model_df["StationId"].nunique()
    ),

    "model": "Random Forest",

    "offline_mae": float(
        rf_mae_10
    ),

    "offline_rmse": float(
        rf_rmse_10
    ),

    "offline_r2": float(
        rf_r2_10
    ),

    "live_style_mae": float(
        live_style_mae
    ),

    "live_style_rmse": float(
        live_style_rmse
    ),

    "live_style_r2": float(
        live_style_r2
    ),

    "prediction_parity_passed": bool(
        parity_passed
    )
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        phase12_status,
        f,
        indent=4
    )

print(
    "Status saved."
)

print(
    "Parity passed:",
    parity_passed
)

In [ ]:
import os
import json
import joblib
import pandas as pd

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save canonical 10-station modeling dataset
# --------------------------------------------------

canonical_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_10_station_feature_engineered.parquet"
)

model_df.to_parquet(
    canonical_dataset_path,
    index=False
)

# --------------------------------------------------
# 2. Save final 10-station Random Forest
# --------------------------------------------------

rf_model_10_path = os.path.join(
    MODEL_PATH,
    "final_10_station_random_forest.joblib"
)

joblib.dump(
    rf_model_10,
    rf_model_10_path
)

# --------------------------------------------------
# 3. Save prediction parity results
# --------------------------------------------------

parity_results_path = os.path.join(
    TABLE_PATH,
    "phase12_0_prediction_parity.csv"
)

parity_results.to_csv(
    parity_results_path,
    index=False
)

# --------------------------------------------------
# 4. Save live-style historical evaluation
# --------------------------------------------------

live_eval_path = os.path.join(
    TABLE_PATH,
    "phase12_0_live_style_evaluation.csv"
)

live_eval.to_csv(
    live_eval_path,
    index=False
)

# --------------------------------------------------
# 5. Save persistence baseline metrics
# --------------------------------------------------

baseline_metrics = {
    "MAE": float(baseline_mae_10),
    "RMSE": float(baseline_rmse_10),
    "R2": float(baseline_r2_10)
}

with open(
    os.path.join(
        OUTPUT_PATH,
        "phase12_0_persistence_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        baseline_metrics,
        f,
        indent=4
    )

# --------------------------------------------------
# 6. Save Random Forest metrics
# --------------------------------------------------

rf_metrics = {
    "MAE": float(rf_mae_10),
    "RMSE": float(rf_rmse_10),
    "R2": float(rf_r2_10)
}

with open(
    os.path.join(
        OUTPUT_PATH,
        "phase12_0_random_forest_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        rf_metrics,
        f,
        indent=4
    )

# --------------------------------------------------
# 7. Save live-style metrics
# --------------------------------------------------

live_metrics = {
    "MAE": float(live_style_mae),
    "RMSE": float(live_style_rmse),
    "R2": float(live_style_r2),
    "Cases": int(len(live_eval))
}

with open(
    os.path.join(
        OUTPUT_PATH,
        "phase12_0_live_style_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        live_metrics,
        f,
        indent=4
    )

# --------------------------------------------------
# 8. Save parity decision
# --------------------------------------------------

parity_max_difference = (
    valid_parity[
        "PredictionDifference"
    ].max()
)

parity_passed = (
    len(valid_parity) > 0
    and parity_max_difference <= 1e-8
)

phase12_summary = {
    "completed_phase": (
        "Phase 12.0 - 10-Station Training/Inference Parity Audit"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "station_count": 10,

    "model": "Random Forest",

    "model_rows": int(
        len(model_df)
    ),

    "offline": {
        "MAE": float(rf_mae_10),
        "RMSE": float(rf_rmse_10),
        "R2": float(rf_r2_10)
    },

    "live_style_historical": {
        "MAE": float(live_style_mae),
        "RMSE": float(live_style_rmse),
        "R2": float(live_style_r2),
        "Cases": int(len(live_eval))
    },

    "prediction_parity": {
        "Cases": int(len(valid_parity)),
        "MaximumDifference": float(
            parity_max_difference
        ),
        "Passed": bool(
            parity_passed
        )
    }
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase12_0_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase12_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 9. Save deployment configuration
# --------------------------------------------------

deployment_config = {
    "model": "Random Forest",
    "model_file": "final_10_station_random_forest.joblib",
    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",
    "forecast_horizon_hours": 1,

    "canonical_feature_builder": (
        "build_canonical_features"
    ),

    "prediction_parity_passed": bool(
        parity_passed
    ),

    "live_api_connected": False
}

deployment_path = os.path.join(
    OUTPUT_PATH,
    "phase12_0_deployment_config.json"
)

with open(
    deployment_path,
    "w"
) as f:
    json.dump(
        deployment_config,
        f,
        indent=4
    )

# --------------------------------------------------
# 10. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": (
        "Phase 12.0 - 10-Station Training/Inference Parity Audit"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "current_model": "Random Forest",
    "station_count": 10,
    "parity_passed": bool(
        parity_passed
    )
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 11. Verify
# --------------------------------------------------

files_to_check = [
    canonical_dataset_path,
    rf_model_10_path,
    parity_results_path,
    live_eval_path,
    summary_path,
    deployment_path,
    status_path
]

print("=" * 60)
print("PHASE 12.0 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nParity passed:", parity_passed)
print("Current model: Random Forest")
print("Stations: 10")
print("Next: Phase 13 - CPCB/OGD Live Data Integration")